In [9]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-6"

In [10]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[], output_config=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }
    
    extra_body = {
        "temperature": temperature
    }

    if system:
        params["system"] = system
        
    if output_config:
        params["output_config"] = output_config

    message = client.messages.create(**params, extra_body=extra_body)
    return message.content[0].text

In [11]:
import json

def generate_dataset():
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "task": {"type": "string"},
                        "format": {"type": "string"},
                        "solution_criteria": {"type": "string"},
                    },
                    "required": ["task", "format", "solution_criteria"],
                    "additionalProperties": False
                }
            }
        }
    }
    
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    text = chat(messages, output_config=output_config)
    
    return json.loads(text)

In [12]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [13]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:
    
    {test_case["task"]}
    
    * Respond only with Python, JSON, or a plain Regex
    * Do not add any comments or commentary or explanation
    """
    
    messages = []
    add_user_message(messages, prompt)
    
    output_config = {
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "content": {"type": "string"}
                },
                "required": ["content"],
                "additionalProperties": False,
            }
        }
    }
    
    response = chat(messages, output_config=output_config)
    return json.loads(response)["content"]

In [14]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "strengths": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },
                    "weaknesses": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },
                    "reasoning": {
                        "type": "string"
                    },
                    "score": {
                        "type": "number"
                    },
                    "solution_criteria": {
                        "type": "string"
                    }
                },
                "required": ["strengths", "weaknesses", "reasoning", "score", "solution_criteria"],
                "additionalProperties": False
            }
        }
    }
    
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(messages, output_config=output_config)
    return json.loads(eval_text)

In [15]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0
    
def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0
    
def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [16]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    strengths = model_grade["strengths"]
    weaknesses = model_grade["weaknesses"]
    
    syntax_score = grade_syntax(output, test_case)
    
    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
        "strengths": strengths,
        "weaknesses": weaknesses
    }
    

In [17]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
        
    return results

In [18]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
    
results = run_eval(dataset)

Average score: 8.666666666666666


In [19]:
print(json.dumps(results, indent=2))

[
  {
    "output": "import boto3\n\ndef list_s3_keys(bucket_name: str, prefix: str) -> list[str]:\n    s3 = boto3.client('s3')\n    keys = []\n    paginator = s3.get_paginator('list_objects_v2')\n    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):\n        for obj in page.get('Contents', []):\n            keys.append(obj['Key'])\n    return keys\n",
    "test_case": {
      "task": "Write a Python function that takes an S3 bucket name and a prefix string as arguments, uses boto3 to list all object keys under that prefix, and returns them as a list of strings.",
      "format": "python",
      "solution_criteria": "Function uses boto3 S3 client, handles pagination with list_objects_v2, filters by the given prefix, and returns a flat list of key strings."
    },
    "score": 9.5,
    "reasoning": "The solution fully satisfies all stated criteria: it uses a boto3 S3 client, handles pagination with list_objects_v2, filters by the given prefix, and returns a flat list of